In [1]:
import pandas as pd
import numpy as np
df = pd.read_csv('data/Telco-Customer-Churn.csv')

In [33]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [30]:
mean = df['TotalCharges'].mean()
std = df['TotalCharges'].std()
cv = std/mean
print(cv)

0.9943239728179847


In [31]:
q1 = df['TotalCharges'].quantile(0.25)
q3 = df['TotalCharges'].quantile(0.75)
iqr = q3-q1
print(iqr)

3388.0499999999997


In [32]:
lower_boundary = q1-(iqr*1.5)
upper_boundary = q3+(iqr*1.5)
print(lower_boundary)
print(upper_boundary)

-4683.525
8868.675


In [22]:
df['TotalCharges'].head()

0      29.85
1     1889.5
2     108.15
3    1840.75
4     151.65
Name: TotalCharges, dtype: str

In [2]:
df['TotalCharges'] = df['TotalCharges'].replace(" ",np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

In [29]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692,2279.734304
std,0.368612,24.559481,30.090047,2266.794470
min,0.000000,0.000000,18.250000,0.000000
25%,0.000000,9.000000,35.500000,398.550000
50%,0.000000,29.000000,70.350000,1394.550000
75%,0.000000,55.000000,89.850000,3786.600000
max,1.000000,72.000000,118.750000,8684.800000


In [3]:
df[df['TotalCharges'].isnull()][['tenure','MonthlyCharges']]

,tenure,MonthlyCharges
488,0,52.55
753,0,20.25
936,0,80.85
1082,0,25.75
1340,0,56.05
3331,0,19.85
3826,0,25.35
4380,0,20.00
5218,0,19.70
6670,0,73.35


In [4]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [35]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [5]:
df['customerID'].nunique()
df = df.drop('customerID',axis = 1)

In [6]:
df['gender'] = df['gender'].map({'Male': 0, 'Female': 1})
df['Partner'] = df['Partner'].map({'No': 0, 'Yes': 1})
df['Dependents'] = df['Dependents'].map({'No': 0, 'Yes': 1})
df['PhoneService'] = df['PhoneService'].map({'No': 0, 'Yes': 1})
df['PaperlessBilling'] = df['PaperlessBilling'].map({'No': 0, 'Yes': 1})
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})

In [7]:
df = pd.get_dummies(df, columns=[
    'MultipleLines',
    'InternetService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'PaymentMethod'
], drop_first=True)

In [8]:
df['Contract'] = df['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})

In [9]:
df.corr(numeric_only=True)['Churn'].sort_values(ascending=False)

Churn                                    1.000000
InternetService_Fiber optic              0.308020
PaymentMethod_Electronic check           0.301919
MonthlyCharges                           0.193356
PaperlessBilling                         0.191825
SeniorCitizen                            0.150889
StreamingTV_Yes                          0.063228
StreamingMovies_Yes                      0.061382
MultipleLines_Yes                        0.040102
PhoneService                             0.011942
gender                                   0.008612
MultipleLines_No phone service          -0.011942
DeviceProtection_Yes                    -0.066160
OnlineBackup_Yes                        -0.082255
PaymentMethod_Mailed check              -0.091683
PaymentMethod_Credit card (automatic)   -0.134302
Partner                                 -0.150448
Dependents                              -0.164221
TechSupport_Yes                         -0.164674
OnlineSecurity_Yes                      -0.171226


In [10]:
X = df.drop('Churn',axis=1)
y = df['Churn']

In [11]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [18]:
models = {
    'LogisticRegression':LogisticRegression(),
    'DecisionTreeC':DecisionTreeClassifier(max_depth=4,random_state=42),
    'RandomForestC':RandomForestClassifier(random_state=42),
    'KNeighborsC':KNeighborsClassifier(),
    'SVC':SVC(random_state=42),
    'NaiveBayes':GaussianNB(),
    'XGBoost':XGBClassifier(random_state=42)
}

for name,model in models.items():
    try:
        model.fit(X_train_scaled,y_train)
        predictions = model.predict(X_test_scaled)

        print(f"--- {name} ---")
        print("Accuracy:", accuracy_score(y_test, predictions))
        print("Precision:", precision_score(y_test, predictions))
        print("Recall:", recall_score(y_test, predictions))
        print("F1 Score:", f1_score(y_test, predictions))
        print('-------------')
    except Exception as e:
        print(f"--- {name} FAILED ---")
        print(f"Error: {e}")
        print()

--- LogisticRegression ---
Accuracy: 0.8211497515968772
Precision: 0.6850152905198776
Recall: 0.6005361930294906
F1 Score: 0.64
-------------
--- DecisionTreeC ---
Accuracy: 0.7970191625266146
Precision: 0.6570397111913358
Recall: 0.4879356568364611
F1 Score: 0.56
-------------
--- RandomForestC ---
Accuracy: 0.7977288857345636
Precision: 0.6605839416058394
Recall: 0.48525469168900803
F1 Score: 0.5595054095826894
-------------
--- KNeighborsC ---
Accuracy: 0.7700496806245565
Precision: 0.5763239875389408
Recall: 0.4959785522788204
F1 Score: 0.5331412103746398
-------------
--- SVC ---
Accuracy: 0.8133427963094393
Precision: 0.6950354609929078
Recall: 0.5254691689008043
F1 Score: 0.5984732824427481
-------------
--- NaiveBayes ---
Accuracy: 0.6650106458481192
Precision: 0.4351245085190039
Recall: 0.8900804289544236
F1 Score: 0.5845070422535211
-------------
--- XGBoost ---
Accuracy: 0.7977288857345636
Precision: 0.6392405063291139
Recall: 0.5415549597855228
F1 Score: 0.5863570391872278


In [19]:
lr_balanced = LogisticRegression(class_weight='balanced')
lr_balanced.fit(X_train_scaled, y_train)
lr_balanced_predictions = lr_balanced.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, lr_balanced_predictions))
print("Precision:", precision_score(y_test, lr_balanced_predictions))
print("Recall:", recall_score(y_test, lr_balanced_predictions))
print("F1 Score:", f1_score(y_test, lr_balanced_predictions))

Accuracy: 0.7494677075940384
Precision: 0.5168350168350169
Recall: 0.8230563002680965
F1 Score: 0.6349534643226473


In [20]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

balanced_models = {
    'Decision Tree (balanced)': DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced'),
    'Random Forest (balanced)': RandomForestClassifier(random_state=42, class_weight='balanced'),
    'SVM (balanced)': SVC(random_state=42, class_weight='balanced')
}

for name, model in balanced_models.items():
    try:
        model.fit(X_train_scaled, y_train)
        predictions = model.predict(X_test_scaled)
        
        print(f"--- {name} ---")
        print("Accuracy:", accuracy_score(y_test, predictions))
        print("Precision:", precision_score(y_test, predictions))
        print("Recall:", recall_score(y_test, predictions))
        print("F1 Score:", f1_score(y_test, predictions))
        print()
    except Exception as e:
        print(f"--- {name} FAILED ---")
        print(f"Error: {e}")
        print()

--- Decision Tree (balanced) ---
Accuracy: 0.7629524485450674
Precision: 0.5363128491620112
Recall: 0.7721179624664879
F1 Score: 0.6329670329670329

--- Random Forest (balanced) ---
Accuracy: 0.7835344215755855
Precision: 0.5817307692307693
Recall: 0.6487935656836461
F1 Score: 0.6134347275031685

--- SVM (balanced) ---
Accuracy: 0.7537260468417317
Precision: 0.5226480836236934
Recall: 0.8042895442359249
F1 Score: 0.6335797254487856



In [ ]:
from xgboost import XGBClassifier

# Calculate the ratio of negative to positive class (standard recommended approach)
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
weight_ratio = neg_count / pos_count

print("Class weight ratio:", weight_ratio)

xgb_balanced = XGBClassifier(random_state=42, scale_pos_weight=weight_ratio)
xgb_balanced.fit(X_train_scaled, y_train)
xgb_predictions = xgb_balanced.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, xgb_predictions))
print("Precision:", precision_score(y_test, xgb_predictions))
print("Recall:", recall_score(y_test, xgb_predictions))
print("F1 Score:", f1_score(y_test, xgb_predictions))

Class weight ratio: 2.766042780748663
Accuracy: 0.7700496806245565
Precision: 0.5533769063180828
Recall: 0.6809651474530831
F1 Score: 0.6105769230769231
